# AttackAware PolyIoM v1.1.4 — results figures

Builds the four results figures from `runs/scores/score_histograms.json`. One
figure per ISO/IEC 24745 criterion. Read-only: no seal, no threshold, no new
score is computed, so this notebook cannot disagree with the sealed numbers.

| Figure | Criterion | What it shows |
|---|---|---|
| A | performance preservation | protected DET curves, and what protection cost against the unprotected baseline |
| B | separability | genuine and impostor distributions with `tau*` |
| C | unlinkability | mated and non-mated with `D_link(s)` overlaid |
| D | revocability | genuine, impostor and pseudo-impostor together |

## Two honesty devices built into the figures

**Figure A does not draw the voice baseline as a DET curve.** The unprotected
voice system makes fewer than one genuine error: the split resolves 0.17%
(held-out) and 0.09% (external), and the measured EERs are 0.085% and 0.003%.
A curve through that region would imply precision the trial count cannot
support, and the ratio it suggests (22.7x, 848x) is an artifact of dividing by
an unresolved number. The right panel therefore reports EER directly, drawing
the baseline as a one-sided 95% upper bound where it is unresolved and as a
point only where it is genuinely measured. Each row is labelled **firm**,
**not resolvable**, or **no interval**, so the figure states which contrasts
the data settle.

**Figure C shades the part of its own curve that is unreliable.** `D_link(s)`
is a density ratio, and the mated density comes from one comparison per
identity. Where the non-mated histogram holds fewer than five pairs the ratio
is driven by single observations and runs to 1 for that reason alone. Those
bins are shaded rather than silently plotted.

## One caution to carry into the paper

The mated distribution in figure C and the pseudo-impostor distribution in
figure D are **the same measurement**. The paper must not present
unlinkability and revocability as independent evidence.

## Colour means one thing throughout

blue = genuine or protected · grey = impostor or non-mated ·
green = pseudo-impostor or face · orange = the unprotected baseline, and only
ever that.

Writes eight files (four figures, PDF and PNG) to `figures/results/`, and
then packages them with their explanation as
`figures/Results_Figures_Package.docx`. The explanation text comes from
`results_figures_text.json`, which the local builder reads too, so the
two cannot drift apart.


In [ ]:
#@title 1. Mount Drive and verify the figure inputs
from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/AttackAware_PolyIoM_v1_1_4")
DIR = {"runs": PROJECT / "runs", "seal": PROJECT / "seal"}

SCORES = DIR["runs"] / "scores" / "score_histograms.json"
if not SCORES.is_file():
    raise FileNotFoundError(
        f"{SCORES} is missing. Run the SCOREDUMP notebook first; this "
        f"notebook only draws what that one measured."
    )

data = json.loads(SCORES.read_text())
if data.get("SYNTHETIC"):
    raise ValueError("This dump is synthetic test data, not the real run.")

print("Preflight: PASS")
for modality, entry in data["modalities"].items():
    parts = ", ".join(entry["partitions"])
    print(f"  {modality:<6} M={entry['M']:<4} tau={entry['tau']:<4} {parts}")
print("\nThis notebook writes only figures. It changes no seal.")

In [ ]:
#@title 2. Load the figure runtime
path = PROJECT / "figures_only.py"
exec(compile(path.read_text(), str(path), "exec"), globals())

In [ ]:
#@title 3. Build the four results figures
outdir = make_figures()

from IPython.display import Image, display
for stem in ("figA_det_preservation", "figB_score_separability",
             "figC_unlinkability", "figD_revocability"):
    display(Image(filename=str(outdir / f"{stem}.png"), width=900))

In [ ]:
#@title 4. Package the figures and their explanation as a .docx
!pip -q install python-docx

path = PROJECT / "docx_only.py"
exec(compile(path.read_text(), str(path), "exec"), globals())
package = build_results_docx()